In [15]:
import os
import pandas as pd
import matplotlib.pyplot as plt

In [16]:
folder_path = './evaluation_results'
files = list()

for filename in os.listdir(folder_path):
    if filename.endswith(".csv"):
        full_path = os.path.join(folder_path, filename)
        files.append(full_path)

In [17]:
df = None
for file in files:
    temp_df = pd.read_csv(file)
    if "saits" in file.lower():
        temp_df['model'] = 'SAITS'
    elif "brits" in file.lower():
        temp_df['model'] = 'BRITS'
    elif "gpvae" in file.lower():
        temp_df['model'] = 'GPVAE'
    elif "csdi" in file.lower():
        temp_df['model'] = 'CSDI'
    df = pd.concat([df, temp_df], ignore_index=True)


In [18]:
print(df)

                  Feature        MAE        RMSE  Missing_Rate  model
0       OVERALL (Average)   6.615874   98.776677           0.3  SAITS
1                p (mbar)   0.324778    0.422604           0.3  SAITS
2                T (degC)   0.315796    0.386117           0.3  SAITS
3                Tpot (K)   0.304508    0.376089           0.3  SAITS
4             Tdew (degC)   0.237608    0.293465           0.3  SAITS
..                    ...        ...         ...           ...    ...
259           SWDR (W/m�)  13.404795   36.403287           0.5  BRITS
260       PAR (�mol/m�/s)  24.144212   68.840158           0.5  BRITS
261  max. PAR (�mol/m�/s)  56.420687  303.068387           0.5  BRITS
262           Tlog (degC)   0.375108    0.749857           0.5  BRITS
263                    OT  13.642352  303.418441           0.5  BRITS

[264 rows x 5 columns]


In [20]:
def create_comparison_table(df, missing_rate):
    subset = df[df['Missing_Rate'] == missing_rate].copy()

    if subset.empty:
        return None
    pivot_df = subset.pivot(index='Feature', columns='model', values=['MAE', 'RMSE'])
    # Swap levels to have Model first, Metric second
    pivot_df.columns = pivot_df.columns.swaplevel(0, 1)
    # Same Model together
    pivot_df.sort_index(axis=1, inplace=True)

    return pivot_df

In [21]:
rates = [0.3, 0.5, 0.7]
result_tables = {}
for rate in rates:
    table = create_comparison_table(df, rate)
    if table is not None:
        result_tables[rate] = table
        print(f"\n=== Missing Rate: {rate} 比較表 ===")
        print(table)

        table.to_csv(f"comparison_{rate}.csv")
    else:
        print(f"\n沒有 Missing Rate: {rate} 的資料")


=== Missing Rate: 0.3 比較表 ===
model                     BRITS                   CSDI              \
                            MAE        RMSE        MAE        RMSE   
Feature                                                              
H2OC (mmol/mol)        0.033907    0.118627   0.075707    0.155478   
OT                    11.539096  299.967915  11.642276  308.640728   
OVERALL (Average)      6.029529   95.682797   6.615558  101.016644   
PAR (�mol/m�/s)       15.590149   50.863569  15.923265   57.414256   
SWDR (W/m�)            8.579874   26.881152   8.581353   30.151303   
T (degC)               0.060335    0.220917   0.122826    0.180566   
Tdew (degC)            0.131907    0.311125   0.111364    0.182520   
Tlog (degC)            0.159138    0.546954   0.212044    0.305056   
Tpot (K)               0.066679    0.219845   0.129023    0.189531   
VPact (mbar)           0.040157    0.125255   0.073325    0.149608   
VPdef (mbar)           0.069283    0.264229   0.162534    0